# Ch 15 — 보스턴 집값 예측 (회귀)

원본: `13_Boston.py`

다루는 내용:
1. 공백 구분 CSV 로드
2. 13개 입력 → 1개 실수 출력
3. MSE 손실 + MAE 지표
4. 예측값 vs 실제값 비교

## 0. 환경

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import Input, Sequential
from keras.layers import Dense

keras.utils.set_random_seed(0)

print("Keras:", keras.__version__)

from sklearn.model_selection import train_test_split

## 1. 데이터 로드

> ⚠ 보스턴 집값 데이터는 컬럼 `B` (인종 관련) 때문에 sklearn 1.2부터 제거됨.
> 본 노트북은 *학습 목적의 토이 예제*로만 사용.

In [ ]:
DATA = "../../data/housing.csv"
df = pd.read_csv(DATA, sep=r"\s+", header=None)  # ← delim_whitespace 대체
print("shape:", df.shape)
df.head()

컬럼 의미 (참고):

| # | 이름 | 의미 |
| - | - | - |
| 0 | CRIM | 인당 범죄율 |
| 1 | ZN | 25000 sq.ft 이상 거주지역 비율 |
| 2 | INDUS | 비소매업 면적 비율 |
| 3 | CHAS | 찰스강 인접 (0/1) |
| 4 | NOX | 일산화질소 농도 |
| 5 | RM | 주택당 평균 방 수 |
| 6 | AGE | 1940년 이전 건축 비율 |
| 7 | DIS | 5개 직업센터까지의 가중거리 |
| 8 | RAD | 고속도로 접근성 지수 |
| 9 | TAX | 재산세율 |
| 10 | PTRATIO | 학생/교사 비율 |
| 11 | B | (인종 관련 — 윤리적 이유로 비권장) |
| 12 | LSTAT | 저소득층 비율 |
| 13 | MEDV | **집값 중간값 (천 달러)** ← 라벨 |

## 2. 입력/라벨 분리 + train/test

In [ ]:
X = df.iloc[:, 0:13].to_numpy(dtype="float32")
y = df.iloc[:, 13].to_numpy(dtype="float32")

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
print("train:", X_tr.shape, "test:", X_te.shape)
print(f"y 범위: {y.min():.1f} ~ {y.max():.1f}  (단위: 천 달러)")

## 3. 모델 — 출력층은 활성화 없음

회귀에서 출력층에 sigmoid/softmax를 쓰면 출력 범위가 제한되어 안 됨. 그냥 선형(`Dense(1)`).

In [ ]:
keras.utils.set_random_seed(0)
model = Sequential([
    Input(shape=(13,)),
    Dense(30, activation="relu"),
    Dense(6,  activation="relu"),
    Dense(1),  # ← 활성화 없음
])
model.compile(loss="mean_squared_error", optimizer="adam", metrics=["mae"])
model.summary()

## 4. 학습

In [ ]:
hist = model.fit(X_tr, y_tr, epochs=200, batch_size=10,
                 validation_data=(X_te, y_te), verbose=0)

mse = hist.history["loss"][-1]
val_mse = hist.history["val_loss"][-1]
mae = hist.history["mae"][-1]
val_mae = hist.history["val_mae"][-1]
print(f"train  MSE = {mse:.2f}  MAE = {mae:.2f}")
print(f"val    MSE = {val_mse:.2f}  MAE = {val_mae:.2f}")
print(f"\n평균적으로 약 {val_mae:.1f} (천 달러) 정도 빗나감")

## 5. 학습 곡선

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.subplot(1, 2, 1)
plt.plot(hist.history["loss"], label="train"); plt.plot(hist.history["val_loss"], label="val")
plt.title("MSE"); plt.legend(); plt.grid(alpha=0.3)
plt.subplot(1, 2, 2)
plt.plot(hist.history["mae"], label="train"); plt.plot(hist.history["val_mae"], label="val")
plt.title("MAE"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. 예측값 vs 실제값

In [ ]:
y_pred = model.predict(X_te, verbose=0).flatten()

plt.figure(figsize=(5, 5))
plt.scatter(y_te, y_pred, s=20, alpha=0.6)
m = max(y.max(), y_pred.max()) + 5
plt.plot([0, m], [0, m], "k--", lw=1, label="y = x")
plt.xlabel("실제 가격 (천 달러)"); plt.ylabel("예측 가격 (천 달러)")
plt.title("Boston — 예측 vs 실제"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
print(f"{'실제':>8}  {'예측':>8}  {'오차':>8}")
for actual, pred in zip(y_te[:10], y_pred[:10]):
    err = pred - actual
    print(f"{actual:8.2f}  {pred:8.2f}  {err:+8.2f}")

## 마무리
- [ ] 분류 vs 회귀 — 모델 끝단/손실 차이
- [ ] MAE 가 약 3 (천 달러) ⇒ 평균 ~3000 달러 정도 빗나감, 데이터의 평균값 22 대비 어느 정도 좋은지
- [ ] 입력 정규화 (StandardScaler) 를 추가하면 결과가 어떻게 변하는지 (책에는 없음)